## Compound Flood Risk Analysis — Snowmelt (SWE) & Soil Moisture maps
CESM2-LE vs ERA5-interpolated, same methodology as in `create_precip_maps_hans.ipynb`
(2-day median + 90th-percentile + percentile significance testing + seasonal analysis + diagnostics for testing). Snowmelt = multi-day melt; soil moisture = multi-day rolling mean. Only 90 of the 100 CESM2-LE
members for SWE/soil-moisture, so ensemble size is auto-detected (n=90).
Output → `figures/compound_flood_risk_output/`.


### Time-Period Selector for the Whole Analysis

In [1]:
# %% [Setup, imports, per-variable configuration]
import os
import sys
from pathlib import Path

# ── Thread settings: set before importing numpy/xarray/etc. ───────────────────
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

# ── User settings: keep these near the top ────────────────────────────────────
HELPER_DIR     = Path("/nird/home/lbal/internship_storm_hans/helper")

FIG_SUBDIR     = "compound_flood_risk_output"
MAP_START      = 1995
MAP_END        = 2024
MAP_EXTENT_ANN = (5.0, 14.0, 57.5, 64.0)

RECOMPUTE      = True   # True → rebuild ALL caches; slow
WINDOW_DAYS    = 2      # 1/2/3/4-day window
WLABEL         = f"{WINDOW_DAYS}-Day"

# ── Make helper modules importable ────────────────────────────────────────────
if not HELPER_DIR.exists():
    raise FileNotFoundError(f"Helper directory does not exist: {HELPER_DIR}")

if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

# Drop ALL cached helper modules so edited .py files take effect on re-run
for _mod in ("config_paths", "catchment_tools", "plot_style",
             "data_era5", "data_senorge", "data_smile", "return_period"):
    sys.modules.pop(_mod, None)

import config_paths as cfg

print("Loaded config_paths from:", cfg.__file__)

# Derived setting that depends on config_paths.py
FSTEM = f"{cfg.acc_tag(WINDOW_DAYS)}median"   # e.g. "2daymedian"

# ── Standard imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.feature as cfeature  # used by plot_style internals
import dask

dask.config.set(scheduler="synchronous")

# ── Helper imports ────────────────────────────────────────────────────────────
from plot_style import (
    plot_window_interp_3panel,
    plot_window_interp_diffonly,
    plot_window_interp_diffonly_sig,
    plot_window_interp_seasonal_4row_3col,
)

from data_era5 import (
    compute_era5_interpolated_window_median_2d,
    compute_era5_interpolated_window_p90_2d,
    compute_era5_interpolated_window_seasonal_median_2d,
    compute_era5_interpolated_window_seasonal_p90_2d,
)

from data_smile import (
    compute_cesm2_le_window_global_median_2d,
    compute_cesm2_le_window_per_member_p90_2d,
    compute_significance_masks,
    compute_cesm2_le_window_seasonal_global_median_2d,
    compute_cesm2_le_window_seasonal_per_member_p90_2d,
)

from catchment_tools import (
    load_catchments,
    rolling_change,
    rolling_melt,
    rolling_mean,
    rolling_identity,
    subset_time_series_by_year,
    open_field_cache,
)

# ── Catchment labels ──────────────────────────────────────────────────────────
CATCHMENT_NUMBERS = {
    "nevina_bergheim":  1,
    "nevina_honnefoss": 2,
    "nevina_losna":     3,
    "regine_drammen":   4,
    "regine_glomma":    5,
}

CATCHMENT_LEGEND_TEXT = (
    "Catchments:\n"
    "  1 · Nevina Bergheim\n"
    "  2 · Nevina Hønnefoss\n"
    "  3 · Nevina Losna\n"
    "  4 · Regine Drammen\n"
    "  5 · Regine Glomma"
)

CATCHMENT_LABEL_OVERRIDES = {
    "nevina_bergheim": (8.3, 60.8),
    "regine_drammen":  (9.8, 60.2),
}

SIG_LEGEND_5_95 = (
    "Significance (p=0.10, 2-sided):\n"
    "  //  CESM2-LE > ERA5-Interp.\n"
    "  \\\\ ERA5-Interp. > CESM2-LE"
)

SIG_LEGEND_2_98 = (
    "Significance (p=0.04, 2-sided):\n"
    "  //  CESM2-LE > ERA5-Interp.\n"
    "  \\\\ ERA5-Interp. > CESM2-LE"
)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.dpi": 150,
})

# ── Dependency callables passed into helper functions ─────────────────────────
_sub           = subset_time_series_by_year
_roll_change   = rolling_change
_roll_melt     = rolling_melt
_roll_mean     = rolling_mean
_roll_identity = rolling_identity

def _open_field(kind):
    return lambda p, s, e: open_field_cache(p, kind, s, e)

def figp(fname):
    return cfg.precip_map_figure_paths(FIG_SUBDIR, fname)

# ── Per-variable configuration ────────────────────────────────────────────────
VARIABLES = {
    "snowmelt": dict(
        kind="swe",
        var_word="snowmelt",
        noun="Snowmelt",
        cesm2_dir=cfg.CESM2_LE_SWE_DIR,
        era5_interp_dir=cfg.ERA5_INTERPOLATED_SWE_DIR,
        cesm2_raw_var="SWE",
        era5_raw_var="sd",
        roll=_roll_identity,
        open=_open_field("swe"),
        daily_window=WINDOW_DAYS,
        median_vmax=5.0,
        p90_vmax=20.0,
        seq_label_median="ΔSWE Median (kg/m²)",
        seq_label_p90="90th-pctl. ΔSWE (kg/m²)",
        div_label_median="Difference of ΔSWE Median (%),\nCESM2-LE – ERA5 Interpolated",
        div_label_p90="90th-pctl. ΔSWE Difference (%),\nCESM2-LE – ERA5 Interpolated",
        title_median="Snowmelt (ΔSWE) Median Difference",
        title_p90="90th-Percentile Snowmelt (ΔSWE) Difference",
        title_seasonal_median="Seasonal Snowmelt (ΔSWE) Median Difference",
        title_seasonal_p90="Seasonal 90th-Percentile Snowmelt (ΔSWE) Difference",
        diag_med_label="median ΔSWE = SWE(t)−SWE(t−1) (kg/m²)",
        diag_p90_label="90th pctl. ΔSWE = SWE(t)−SWE(t−1) (kg/m²)",
    ),

    "soil_moisture": dict(
        kind="soil_moisture",
        var_word="soil_moisture",
        noun="Soil Moisture",
        cesm2_dir=cfg.CESM2_LE_SM_DIR,
        era5_interp_dir=cfg.ERA5_INTERPOLATED_SWVL_DIR,
        cesm2_raw_var="SM",
        era5_raw_var="swvl",
        roll=_roll_mean,
        open=_open_field("soil_moisture"),
        daily_window=1,
        median_vmax=3500.0,
        p90_vmax=3500.0,
        seq_label_median="Soil Moisture Median (kg/m²)",
        seq_label_p90="90th-pctl. Soil Moisture (kg/m²)",
        div_label_median="Difference of Soil Moisture Median (%),\nCESM2-LE – ERA5 Interpolated",
        div_label_p90="90th-pctl. Soil Moisture Difference (%),\nCESM2-LE – ERA5 Interpolated",
        title_median="Soil Moisture Median Difference",
        title_p90="90th-Percentile Soil Moisture Difference",
        title_seasonal_median="Seasonal Soil Moisture Median Difference",
        title_seasonal_p90="Seasonal 90th-Percentile Soil Moisture Difference",
        diag_med_label="daily SM median (kg/m²)",
        diag_p90_label="90th pctl. daily SM (kg/m²)",
    ),
}

print("Loading catchment polygons ...")
catchments_maps = load_catchments(cfg.GEOJSON_FILES, cfg.GEOJSON_DIR)

STORE, SEASONAL_MED, SEASONAL_P90 = {}, {}, {}

print("Setup done.")

Loaded config_paths from: /nird/home/lbal/internship_storm_hans/helper/config_paths.py
Loading catchment polygons ...
Setup done.


### Window Cache for Check if SWE Difference Data is Available

In [2]:
# %% [SWE ΔSWE window-cache availability check — fail early if the cache is missing]
# The N-day signed-snowmelt ΔSWE caches (min(0, ΔSWE), gains→0) are pre-built in
# load_data_store_postprocessed.ipynb. Soil moisture reads the raw 1-day cache and
# is rolled on the fly, so only SWE needs this guard. If the WINDOW_DAYS cache is
# absent, this raises with instructions to build it first.
from data_smile import find_smile_members, get_year_range_smile
from data_era5  import find_era5_interpolated_files, get_year_range_era5_interp

def _require_swe_window_caches(window_days: int) -> None:
    if window_days < 2:
        raise ValueError(
            f"WINDOW_DAYS={window_days}: snowmelt ΔSWE needs a window of >= 2 days "
            "(SWE(t) − SWE(t−(N−1))). Set WINDOW_DAYS in {2, 3, 4}.")
    missing = []
    c_s, c_e = get_year_range_smile(cfg.CESM2_LE_SWE_DIR, "cesm2_le")
    members  = find_smile_members(cfg.CESM2_LE_SWE_DIR, "cesm2_le")
    for mid in members:
        p = cfg.field_daily_cache_path("cesm2_le", "swe", c_s, c_e,
                                       member_id=mid, window_days=window_days)
        if not p.exists():
            missing.append(p)
    e_s, e_e = get_year_range_era5_interp(find_era5_interpolated_files(cfg.ERA5_INTERPOLATED_SWE_DIR))
    p_e = cfg.field_daily_cache_path("era5_interpolated", "swe", e_s, e_e, window_days=window_days)
    if not p_e.exists():
        missing.append(p_e)
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} {window_days}-day ΔSWE snowmelt cache(s) are missing, e.g.:\n"
            f"    {missing[0]}\n"
            f"→ Build them first: in load_data_store_postprocessed.ipynb set "
            f"WINDOW_DAYS_SWE={window_days} in the 'N-day ΔSWE snowmelt cache builder' "
            "cell and run it, then re-run this notebook.")
    print(f"[ok] All {window_days}-day ΔSWE snowmelt caches present "
          f"({len(members)} CESM2-LE members + ERA5-interp).")

_require_swe_window_caches(WINDOW_DAYS)

[ok] All 2-day ΔSWE snowmelt caches present (90 CESM2-LE members + ERA5-interp).


### Load once and store in post-processed folder SWE and Soil Moisture Data

In [3]:
# %% [Seasonal N-day median + p90 computation — cache builder]
for _vkey, _V in VARIABLES.items():
    print(f"\n{'='*60}\n{_V['noun']} — seasonal computation\n{'='*60}")
    k = _V["kind"]
    _dw = _V["daily_window"]   # SWE → N-day ΔSWE cache (pre-computed); soil moisture → 1-day raw
    _md = (lambda mid, s, e, k=k, dw=_dw: cfg.field_daily_cache_path("cesm2_le", k, s, e, member_id=mid, window_days=dw))
    _ed = (lambda ds, res, s, e, k=k, dw=_dw: cfg.field_daily_cache_path("era5_interpolated", k, s, e, window_days=dw))

    SEASONAL_MED[_vkey], SEASONAL_P90[_vkey] = {}, {}

    for _s in cfg.SEASONS_ORDER:
        # ── seasonal median ──
        _pm_med_path = cfg.field_window_cache_path("cesm2_le", k, "per_member_medians", WINDOW_DAYS, MAP_START, MAP_END, season=_s)
        da_c_med, _ = compute_cesm2_le_window_seasonal_global_median_2d(
            _s, MAP_START, MAP_END, _V["cesm2_dir"], _md,
            cfg.field_window_cache_path("cesm2_le", k, "global_median", WINDOW_DAYS, MAP_START, MAP_END, season=_s),
            _pm_med_path,
            _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)
        da_e_med = compute_era5_interpolated_window_seasonal_median_2d(
            _s, MAP_START, MAP_END, _V["era5_interp_dir"],
            cfg.field_window_cache_path("era5_interpolated", k, "median", WINDOW_DAYS, MAP_START, MAP_END, season=_s),
            _ed, _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)
        _safe_med = da_e_med.where(da_e_med != 0)
        da_diff_med = (da_c_med - da_e_med) / _safe_med * 100.0
        SEASONAL_MED[_vkey][_s] = dict(per_member_path=_pm_med_path,
                                       da_cesm2=da_c_med, da_era5_interp=da_e_med,
                                       da_diff=da_diff_med)

        # ── seasonal 90th percentile ──
        _pm_p90_path = cfg.field_window_cache_path("cesm2_le", k, "per_member_p90", WINDOW_DAYS, MAP_START, MAP_END, season=_s)
        da_c_p90, _ = compute_cesm2_le_window_seasonal_per_member_p90_2d(
            _s, MAP_START, MAP_END, _V["cesm2_dir"], _md,
            _pm_p90_path,
            cfg.field_window_cache_path("cesm2_le", k, "global_p90", WINDOW_DAYS, MAP_START, MAP_END, season=_s),
            _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)
        da_e_p90 = compute_era5_interpolated_window_seasonal_p90_2d(
            _s, MAP_START, MAP_END, _V["era5_interp_dir"],
            cfg.field_window_cache_path("era5_interpolated", k, "p90", WINDOW_DAYS, MAP_START, MAP_END, season=_s),
            _ed, _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)
        _safe_p90 = da_e_p90.where(da_e_p90 != 0)
        da_diff_p90 = (da_c_p90 - da_e_p90) / _safe_p90 * 100.0
        SEASONAL_P90[_vkey][_s] = dict(per_member_path=_pm_p90_path,
                                       da_cesm2=da_c_p90, da_era5_interp=da_e_p90,
                                       da_diff=da_diff_p90)
print("\nDone (seasonal median + p90 computation).")



Snowmelt — seasonal computation
  Computing CESM2-LE seasonal DJF 90-member 2-day global median (1995–2024) ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    10/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    20/90 members processed ...


...

    70/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    80/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


KeyboardInterrupt: 

### Create 2-day median + 90th pctl. difference + diffonly plots for SWE and Soil Moisture

In [ ]:
# %% [N-day median + 90th-pctile 3-panel diff + diffonly — both variables]
for _vkey, _V in VARIABLES.items():
    print(f"\n{'='*60}\n{_V['noun']}\n{'='*60}")
    k, vw, fstem = _V["kind"], _V["var_word"], FSTEM
    _dw = _V["daily_window"]   # SWE → N-day ΔSWE cache (pre-computed); soil moisture → 1-day raw
    _md = (lambda mid, s, e, k=k, dw=_dw: cfg.field_daily_cache_path("cesm2_le", k, s, e, member_id=mid, window_days=dw))
    _ed = (lambda ds, res, s, e, k=k, dw=_dw: cfg.field_daily_cache_path("era5_interpolated", k, s, e, window_days=dw))


    da_era5_med = compute_era5_interpolated_window_median_2d(
        MAP_START, MAP_END, _V["era5_interp_dir"], _ed, _V["open"], _V["roll"], _sub,
        window_days=WINDOW_DAYS)

    da_cesm2_med = compute_cesm2_le_window_global_median_2d(
        MAP_START, MAP_END, _V["cesm2_dir"], _md,
        cfg.field_window_cache_path("cesm2_le", k, "global_median", WINDOW_DAYS, MAP_START, MAP_END),
        cfg.field_window_cache_path("cesm2_le", k, "per_member_medians", WINDOW_DAYS, MAP_START, MAP_END),
        _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)

    da_cesm2_p90, _ = compute_cesm2_le_window_per_member_p90_2d(
        MAP_START, MAP_END, _V["cesm2_dir"], _md,
        cfg.field_window_cache_path("cesm2_le", k, "per_member_p90", WINDOW_DAYS, MAP_START, MAP_END),
        cfg.field_window_cache_path("cesm2_le", k, "global_p90", WINDOW_DAYS, MAP_START, MAP_END),
        _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)

    da_era5_p90 = compute_era5_interpolated_window_p90_2d(
        MAP_START, MAP_END, _V["era5_interp_dir"],
        cfg.field_window_cache_path("era5_interpolated", k, "p90", WINDOW_DAYS, MAP_START, MAP_END),
        _ed, _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=RECOMPUTE)

    _safe_med = da_era5_med.where(da_era5_med != 0)
    da_diff_med = (da_cesm2_med - da_era5_med) / _safe_med * 100.0
    _safe_p90 = da_era5_p90.where(da_era5_p90 != 0)
    da_diff_p90 = (da_cesm2_p90 - da_era5_p90) / _safe_p90 * 100.0

    STORE[_vkey] = dict(cesm2_med=da_cesm2_med, era5_med=da_era5_med, diff_med=da_diff_med,
                        cesm2_p90=da_cesm2_p90, era5_p90=da_era5_p90, diff_p90=da_diff_p90)

    common = dict(catchments=catchments_maps, start_year=MAP_START, end_year=MAP_END,
                  catchment_numbers=CATCHMENT_NUMBERS, catchment_legend_text=CATCHMENT_LEGEND_TEXT,
                  label_overrides=CATCHMENT_LABEL_OVERRIDES, annmedian_extent=MAP_EXTENT_ANN)

    plot_window_interp_3panel(
        da_cesm2=da_cesm2_med, da_era5_interp=da_era5_med, da_diff=da_diff_med,
        out_paths=figp(f"{fstem}_{vw}_diff_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{WLABEL} {_V['title_median']} ({MAP_START}–{MAP_END})",
        seq_cbar_label=_V["seq_label_median"], div_cbar_label=_V["div_label_median"],
        window_vmax=_V["median_vmax"], **common)
    plot_window_interp_diffonly(
        da_diff=da_diff_med,
        out_paths=figp(f"{fstem}_{vw}_diffonly_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{WLABEL} {_V['title_median']} ({MAP_START}–{MAP_END})",
        div_cbar_label=_V["div_label_median"], **common)
    plot_window_interp_3panel(
        da_cesm2=da_cesm2_p90, da_era5_interp=da_era5_p90, da_diff=da_diff_p90,
        out_paths=figp(f"{fstem}_90pctl_{vw}_diff_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{WLABEL} {_V['title_p90']} ({MAP_START}–{MAP_END})",
        seq_cbar_label=_V["seq_label_p90"], div_cbar_label=_V["div_label_p90"],
        window_vmax=_V["p90_vmax"], **common)
    plot_window_interp_diffonly(
        da_diff=da_diff_p90,
        out_paths=figp(f"{fstem}_90pctl_{vw}_diffonly_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{WLABEL} {_V['title_p90']} ({MAP_START}–{MAP_END})",
        div_cbar_label=_V["div_label_p90"], **common)
print("\nDone (base diff + diffonly figures).")



Snowmelt
  Loading ERA5-interpolated (2-day median) from cache (1995–2024) ...
  Computing CESM2-LE 90-member 2-day GLOBAL median (1995–2024) — this reads all members ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    10/90 members loaded ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    20/90 members loaded ...


...

    70/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    80/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    90/90 members processed ...
  [saved] cesm2_2day_global_p90_soil_moisture_1995-2024.nc
  [saved] cesm2_2day_per_member_p90_soil_moisture_1995-2024.nc
  Computing ERA5-interpolated 2-day p90 (1995–2024) ...
  [saved] 2day_p90_era5interp_soil_moisture_1995-2024.nc
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_soil_moisture_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_soil_moisture_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_soil_moisture_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_soil_moisture_diffonly_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_90pctl_soil_moisture_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_90pctl_soil_moisture_diff_1995-

### Create 2-day median + 90th pctl. difference + diffonly plots with Significance hatching! for SWE and Soil Moisture

In [ ]:
# %% [Significance-hatched N-day median + p90 diff — 5/95 and 2/98 — both variables]
for _vkey, _V in VARIABLES.items():
    print(f"\n{'='*60}\n{_V['noun']} — significance\n{'='*60}")
    S, k, vw, fstem = STORE[_vkey], _V["kind"], _V["var_word"], FSTEM

    with xr.open_dataset(str(cfg.field_window_cache_path("cesm2_le", k, "per_member_medians", WINDOW_DAYS, MAP_START, MAP_END))) as _ds:
        pm_med = _ds[list(_ds.data_vars)[0]].load()
    with xr.open_dataset(str(cfg.field_window_cache_path("cesm2_le", k, "per_member_p90", WINDOW_DAYS, MAP_START, MAP_END))) as _ds:
        pm_p90 = _ds[list(_ds.data_vars)[0]].load()

    common = dict(catchments=catchments_maps, start_year=MAP_START, end_year=MAP_END,
                  catchment_numbers=CATCHMENT_NUMBERS, catchment_legend_text=CATCHMENT_LEGEND_TEXT,
                  label_overrides=CATCHMENT_LABEL_OVERRIDES, annmedian_extent=MAP_EXTENT_ANN)

    for lo, hi, tag, sig_leg in [(5.0, 95.0, "5_95pctl", SIG_LEGEND_5_95),
                                 (2.0, 98.0, "2_98pctl", SIG_LEGEND_2_98)]:
        sc, se = compute_significance_masks(S["era5_med"], pm_med, lower_pctl=lo, upper_pctl=hi)
        plot_window_interp_3panel(
            da_cesm2=S["cesm2_med"], da_era5_interp=S["era5_med"], da_diff=S["diff_med"],
            out_paths=figp(f"{fstem}_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{WLABEL} {_V['title_median']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_median"], div_cbar_label=_V["div_label_median"],
            window_vmax=_V["median_vmax"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
        plot_window_interp_diffonly_sig(
            da_diff=S["diff_med"],
            out_paths=figp(f"{fstem}_{vw}_{tag}_diffonly_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{WLABEL} {_V['title_median']} ({MAP_START}–{MAP_END})",
            div_cbar_label=_V["div_label_median"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
        sc, se = compute_significance_masks(S["era5_p90"], pm_p90, lower_pctl=lo, upper_pctl=hi)
        plot_window_interp_3panel(
            da_cesm2=S["cesm2_p90"], da_era5_interp=S["era5_p90"], da_diff=S["diff_p90"],
            out_paths=figp(f"{fstem}_90pctl_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{WLABEL} {_V['title_p90']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_p90"], div_cbar_label=_V["div_label_p90"],
            window_vmax=_V["p90_vmax"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
        plot_window_interp_diffonly_sig(
            da_diff=S["diff_p90"],
            out_paths=figp(f"{fstem}_90pctl_{vw}_{tag}_diffonly_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{WLABEL} {_V['title_p90']} ({MAP_START}–{MAP_END})",
            div_cbar_label=_V["div_label_p90"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
print("\nDone (significance figures).")



Snowmelt — significance
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_snowmelt_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_snowmelt_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_90pctl_snowmelt_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_outpu

Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_90pctl_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_90pctl_snowmelt_2_98pctl_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_90pctl_snowmelt_2_98pctl_diffonly_1995-2024.pdf

Soil Moisture — significance
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_soil_moisture_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_soil_moisture_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_soil_moisture_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_soil_moisture_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figu

### Diagnostic Tables for Pixels: Overview over Values

In [ ]:
# %% [Diagnostic: pixel-level tables — CESM2-LE vs ERA5-interp (median & p90)]
for _vkey, _V in VARIABLES.items():
    S = STORE[_vkey]
    for _stat, _c, _e, _d in [("median", S["cesm2_med"], S["era5_med"], S["diff_med"]),
                              ("p90",    S["cesm2_p90"], S["era5_p90"], S["diff_p90"])]:
        _lats, _lons = _e["lat"].values, _e["lon"].values
        _cv = _c.sel(lat=_lats, lon=_lons).values
        _ev = _e.values
        _dv = _d.sel(lat=_lats, lon=_lons).values
        _rows = [{"lat": round(float(la), 3), "lon": round(float(lo), 3),
                  "cesm2": round(float(_cv[i, j]), 3), "era5": round(float(_ev[i, j]), 3),
                  "diff_pct": round(float(_dv[i, j]), 1)}
                 for i, la in enumerate(_lats) for j, lo in enumerate(_lons)]
        print(f"\n=== {_V['noun']} — {_stat} (kg/m²) ===")
        print(pd.DataFrame(_rows).to_string(index=False, na_rep="NaN"))



=== Snowmelt — median (kg/m²) ===
   lat   lon  cesm2  era5  diff_pct
56.073  2.50    NaN   0.0       NaN
56.073  3.75    NaN   0.0       NaN
56.073  5.00    NaN   0.0       NaN
56.073  6.25    NaN   0.0       NaN
56.073  7.50    NaN   0.0       NaN
56.073  8.75    0.0   0.0       NaN
56.073 10.00    0.0   0.0       NaN
56.073 11.25    0.0   0.0       NaN
56.073 12.50    0.0   0.0       NaN
56.073 13.75    0.0   0.0       NaN
56.073 15.00    0.0   0.0       NaN
56.073 16.25    0.0   0.0       NaN
57.016  2.50    NaN   0.0       NaN
57.016  3.75    NaN   0.0       NaN
57.016  5.00    NaN   0.0       NaN
57.016  6.25    NaN   0.0       NaN
57.016  7.50    0.0   0.0       NaN
57.016  8.75    0.0   0.0       NaN
57.016 10.00    0.0   0.0       NaN
57.016 11.25    0.0   0.0       NaN
57.016 12.50    0.0   0.0       NaN
57.016 13.75    0.0   0.0       NaN
57.016 15.00    0.0   0.0       NaN
57.016 16.25    0.0   0.0       NaN
57.958  2.50    NaN   0.0       NaN
57.958  3.75    NaN   0.0    

### Seasonal Data Computation for SWE and Soil Moisture

In [ ]:
# %% [Seasonal significance-hatched 4×3 plots — median & p90, 5/95 & 2/98 — both variables]
def _build_seasonal_list(store_for_var, lo, hi):
    out = []
    for _s in cfg.SEASONS_ORDER:
        d = store_for_var[_s]
        with xr.open_dataset(str(d["per_member_path"])) as _ds:
            pm = _ds[list(_ds.data_vars)[0]].load()
        sc, se = compute_significance_masks(d["da_era5_interp"], pm, lower_pctl=lo, upper_pctl=hi)
        out.append({"season_label": cfg.SEASON_LABELS[_s],
                    "da_cesm2": d["da_cesm2"], "da_era5_interp": d["da_era5_interp"],
                    "da_diff": d["da_diff"], "sig_cesm_higher": sc, "sig_era5_higher": se})
    return out

for _vkey, _V in VARIABLES.items():
    print(f"\n{_V['noun']} — seasonal significance plots ...")
    vw, fstem = _V["var_word"], FSTEM
    common = dict(catchments=catchments_maps, start_year=MAP_START, end_year=MAP_END,
                  catchment_numbers=CATCHMENT_NUMBERS, catchment_legend_text=CATCHMENT_LEGEND_TEXT,
                  label_overrides=CATCHMENT_LABEL_OVERRIDES, annmedian_extent=MAP_EXTENT_ANN)
    for lo, hi, tag, sig_leg in [(5.0, 95.0, "5_95pctl", SIG_LEGEND_5_95),
                                 (2.0, 98.0, "2_98pctl", SIG_LEGEND_2_98)]:
        plot_window_interp_seasonal_4row_3col(
            seasonal_data=_build_seasonal_list(SEASONAL_MED[_vkey], lo, hi),
            out_paths=figp(f"{fstem}_seasonal_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{WLABEL} {_V['title_seasonal_median']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_median"], div_cbar_label=_V["div_label_median"],
            window_vmax=_V["median_vmax"], sig_legend_text=sig_leg, **common)
        plot_window_interp_seasonal_4row_3col(
            seasonal_data=_build_seasonal_list(SEASONAL_P90[_vkey], lo, hi),
            out_paths=figp(f"{fstem}_seasonal_90pctl_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{WLABEL} {_V['title_seasonal_p90']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_p90"], div_cbar_label=_V["div_label_p90"],
            window_vmax=_V["p90_vmax"], sig_legend_text=sig_leg, **common)
print("\nDone (seasonal significance figures).")



Snowmelt — seasonal significance plots ...


Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_seasonal_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_seasonal_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_seasonal_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_seasonal_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_seasonal_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_seasonal_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_seasonal_90pctl_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures

### Only Spring Data Computation for SWE and Soil Moisture

In [ ]:
# %% [Spring-only (MAMJ) 3-panel 90th-pctl 2/98 significance diff — annual-figure layout]
# Standalone: reads the existing MAM seasonal p90 caches. Only the setup cell
# (Cell 2) needs to have run — Cell 6 is NOT required.

SPRING      = "MAMJ"                     # cfg.SEASON_LABELS[SPRING] -> "Spring (MAMJ)"
SPRING_PCTL = (2.0, 98.0, "2_98pctl", SIG_LEGEND_2_98)   # keep the '2_98pctl' tag
SPRING_VARS = ["snowmelt"]              # add "soil_moisture" for that variable too

_lo, _hi, _tag, _sig_leg = SPRING_PCTL

for _vkey in SPRING_VARS:
    _V, k = VARIABLES[_vkey], VARIABLES[_vkey]["kind"]
    _dw = _V["daily_window"]
    _md = (lambda mid, s, e, k=k, dw=_dw: cfg.field_daily_cache_path(
        "cesm2_le", k, s, e, member_id=mid, window_days=dw))
    _ed = (lambda ds, res, s, e, k=k, dw=_dw: cfg.field_daily_cache_path(
        "era5_interpolated", k, s, e, window_days=dw))

    print(f"\n{_V['noun']} — loading {SPRING} 90th-pctl fields ...")
    _pm_path = cfg.field_window_cache_path("cesm2_le", k, "per_member_p90",
                                           WINDOW_DAYS, MAP_START, MAP_END, season=SPRING)
    _da_c_p90, _ = compute_cesm2_le_window_seasonal_per_member_p90_2d(
        SPRING, MAP_START, MAP_END, _V["cesm2_dir"], _md,
        _pm_path,
        cfg.field_window_cache_path("cesm2_le", k, "global_p90",
                                    WINDOW_DAYS, MAP_START, MAP_END, season=SPRING),
        _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=False)
    _da_e_p90 = compute_era5_interpolated_window_seasonal_p90_2d(
        SPRING, MAP_START, MAP_END, _V["era5_interp_dir"],
        cfg.field_window_cache_path("era5_interpolated", k, "p90",
                                    WINDOW_DAYS, MAP_START, MAP_END, season=SPRING),
        _ed, _V["open"], _V["roll"], _sub, window_days=WINDOW_DAYS, force_recompute=False)

    _safe = _da_e_p90.where(_da_e_p90 != 0)
    _da_diff_p90 = (_da_c_p90 - _da_e_p90) / _safe * 100.0

    with xr.open_dataset(str(_pm_path)) as _ds:
        _pm = _ds[list(_ds.data_vars)[0]].load()
    _sc, _se = compute_significance_masks(_da_e_p90, _pm, lower_pctl=_lo, upper_pctl=_hi)

    print(f"Plotting {_V['noun']} {SPRING} 90th-pctl {_tag} figure ...")
    plot_window_interp_3panel(
        da_cesm2=_da_c_p90, da_era5_interp=_da_e_p90, da_diff=_da_diff_p90,
        catchments=catchments_maps,
        start_year=MAP_START, end_year=MAP_END,
        out_paths=figp(f"{FSTEM}_{SPRING}_90pctl_{_V['var_word']}_{_tag}_diff_"
                       f"{MAP_START}-{MAP_END}.pdf"),
        fig_title=(f"{cfg.SEASON_LABELS[SPRING]} {WLABEL} {_V['title_p90']} "
                   f"({MAP_START}\u2013{MAP_END})"),
        seq_cbar_label=_V["seq_label_p90"],
        div_cbar_label=_V["div_label_p90"],
        catchment_numbers=CATCHMENT_NUMBERS,
        catchment_legend_text=CATCHMENT_LEGEND_TEXT,
        label_overrides=CATCHMENT_LABEL_OVERRIDES,
        window_vmax=_V["p90_vmax"],
        annmedian_extent=MAP_EXTENT_ANN,
        sig_cesm_higher=_sc, sig_era5_higher=_se,
        sig_legend_text=_sig_leg)
print("\nDone (spring-only significance figure).")



Snowmelt — loading MAMJ 90th-pctl fields ...
  Computing CESM2-LE seasonal MAMJ 90-member 2-day p90 (1995–2024) ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    10/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    20/90 members processed ...


...

    70/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    80/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    90/90 members processed ...
  [saved] cesm2_2day_seasonal_MAMJ_global_p90_swe_1995-2024.nc
  [saved] cesm2_2day_seasonal_MAMJ_per_member_p90_swe_1995-2024.nc
  Computing ERA5-interp seasonal MAMJ 2-day p90 (1995–2024) ...
  [saved] 2day_seasonal_MAMJ_p90_era5interp_swe_1995-2024.nc
Plotting Snowmelt MAMJ 90th-pctl 2_98pctl figure ...
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_MAMJ_90pctl_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_MAMJ_90pctl_snowmelt_2_98pctl_diff_1995-2024.pdf

Done (spring-only significance figure).


### Diagnostics for pixel-cell number 3

In [ ]:
# %% [Seasonal diagnostic: per-season max values for colorbar calibration]
for _vkey, _V in VARIABLES.items():
    print("=" * 72)
    print(f"{_V['noun'].upper()} — SEASONAL MAX VALUES (kg/m²) FOR COLORBAR CALIBRATION")
    print("=" * 72)
    for _name, _store, _vmax in [("median", SEASONAL_MED[_vkey], _V["median_vmax"]),
                                 ("p90",    SEASONAL_P90[_vkey], _V["p90_vmax"])]:
        print(f"\n── {_name} (current vmax used: {_vmax}) ──")
        _allc = []
        for _s in cfg.SEASONS_ORDER:
            d = _store[_s]
            cmax = float(np.nanmax(d["da_cesm2"].values))
            emax = float(np.nanmax(d["da_era5_interp"].values))
            _allc.append(max(cmax, emax))
            print(f"  {cfg.SEASON_LABELS[_s]:<20} CESM2={cmax:>10.3f}  ERA5={emax:>10.3f}")
        print(f"  → suggested vmax: {max(_allc):.1f}")


# %% [Diagnostic: catchment-3 (Losna) grid-cell strip plots — seasonal median & p90]
import geopandas as gpd
_C_CESM2, _C_ERA5 = "#2C7BB6", "#D73027"

_losna_gdf = gpd.read_file(cfg.GEOJSON_DIR / cfg.GEOJSON_FILES["nevina_losna"])
try:
    _pt = _losna_gdf.geometry.union_all().representative_point()
except AttributeError:
    _pt = _losna_gdf.geometry.unary_union.representative_point()
_sel_lon, _sel_lat = _pt.x, _pt.y

for _vkey, _V in VARIABLES.items():
    vw = _V["var_word"]
    for _which, _store, _ylabel, _fname in [
        ("median", SEASONAL_MED[_vkey], _V["diag_med_label"],
        f"diagnostic_catchment3_{FSTEM}_seasonal_{vw}_{MAP_START}-{MAP_END}.pdf"),
        ("p90", SEASONAL_P90[_vkey], _V["diag_p90_label"],
        f"diagnostic_catchment3_{cfg.acc_tag(WINDOW_DAYS)}90pctl_seasonal_{vw}_{MAP_START}-{MAP_END}.pdf"),
    ]:
        _fig, _axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
        _fig.suptitle(f"CESM2-LE ({_V['noun']}) vs ERA5 Interpolated — Catchment 3 (Losna) grid cell\n"
                      f"Seasonal {_which}  ({MAP_START}–{MAP_END})", fontsize=13, y=1.03)
        for _ax, _season in zip(_axes, cfg.SEASONS_ORDER):
            with xr.open_dataset(str(_store[_season]["per_member_path"])) as _ds:
                _pm = _ds[list(_ds.data_vars)[0]]
                _vals = _pm.sel(lat=_sel_lat, lon=_sel_lon, method="nearest").values.ravel()
            _era5v = float(_store[_season]["da_era5_interp"].sel(lat=_sel_lat, lon=_sel_lon, method="nearest"))
            _n = len(_vals)
            _ax.scatter(np.arange(1, _n + 1), _vals, color=_C_CESM2, s=14, alpha=0.65, zorder=2,
                        label=f"CESM2-LE ({_n} members)")
            _ax.scatter(0, _era5v, color=_C_ERA5, s=100, marker="D", zorder=4,
                        label=f"ERA5 Interp. ({_era5v:.2f})")
            _ax.axvline(0.5, color="0.75", lw=0.8, ls="--", zorder=1)
            _ax.set_title(cfg.SEASON_LABELS[_season], fontsize=11)
            _ax.set_xlabel("Member index", fontsize=9); _ax.set_xlim(-3, _n + 2)
            _ax.spines["top"].set_visible(False); _ax.spines["right"].set_visible(False)
        _axes[0].set_ylabel(_ylabel, fontsize=10)
        _axes[-1].legend(fontsize=9, frameon=False, loc="upper right")
        _fig.tight_layout()
        for _root in (cfg.FIGURES_DIR, cfg.FIGURES_DIR_SECONDARY):
            _p = _root / FIG_SUBDIR / _fname
            _p.parent.mkdir(parents=True, exist_ok=True)
            _fig.savefig(str(_p), bbox_inches="tight", dpi=150)
            print(f"Saved → {_p}")
        plt.close(_fig)

SNOWMELT — SEASONAL MAX VALUES (kg/m²) FOR COLORBAR CALIBRATION

── median (current vmax used: 5.0) ──
  Winter (DJF)         CESM2=     0.000  ERA5=     0.011
  Spring (MAM)         CESM2=     2.757  ERA5=     1.656
  Summer (JJA)         CESM2=     0.000  ERA5=     0.000
  Autumn (SON)         CESM2=     0.000  ERA5=     0.000
  → suggested vmax: 2.8

── p90 (current vmax used: 20.0) ──
  Winter (DJF)         CESM2=     5.128  ERA5=     4.356
  Spring (MAM)         CESM2=    16.510  ERA5=    12.619
  Summer (JJA)         CESM2=    15.396  ERA5=     2.417
  Autumn (SON)         CESM2=     0.001  ERA5=     2.221
  → suggested vmax: 16.5
SOIL MOISTURE — SEASONAL MAX VALUES (kg/m²) FOR COLORBAR CALIBRATION

── median (current vmax used: 3500.0) ──
  Winter (DJF)         CESM2=  3219.635  ERA5=  1241.912
  Spring (MAM)         CESM2=  3272.274  ERA5=  1205.364
  Summer (JJA)         CESM2=  3151.340  ERA5=  1163.814
  Autumn (SON)         CESM2=  3124.960  ERA5=  1219.863
  → suggested vm

### Joint-Distribution Ananlysis for Precip - Soil Moisture and Precip. - Snowmelt


In [4]:
# Joint-distribution scatterplot for CESM2-LE catchments
# One point per date and member in the selected range; colours = day of year with a circular Jan–Dec wheel legend 
# Data comes from the caches built in load_data_store_postprocessed.ipynb, cell "# %% [CESM2-LE catchment compound series — run once per window selection]"
# Any unavailable selection (window / years / members) raises an error that states exactly where to build or fix it


from catchment_tools import load_cesm2_le_catchment_field_series
from plot_style import make_joint_distribution_figure

# ── Selection ─────────────────────────────────────────────────────────────────
JD_CATCHMENT     = "regine_drammen_glomma"        # regine_drammen | regine_glomma | regine_drammen_glomma
JD_COMBO         = ("precipitation", "soil_moisture")  # (x, y); also ("precipitation", "snowmelt") or ("precipitation", "soil_moisture") or ("soil_moisture", "precipitation") or ("soil_moisture", "snowmelt") or ("snowmelt", "soil_moisture")
JD_WINDOW_DAYS   = 2                              # select 2-4
JD_START, JD_END = 1995, 2024                     # inside the stored time-period (1920–2034)
JD_MEMBERS       = "all"                          # "all" | "1-30" | [3, 4, 5, 27, 35] or whatever members 

x_var, y_var = JD_COMBO
da_x = load_cesm2_le_catchment_field_series(
    x_var, JD_WINDOW_DAYS, JD_CATCHMENT, JD_START, JD_END, JD_MEMBERS)
da_y = load_cesm2_le_catchment_field_series(
    y_var, JD_WINDOW_DAYS, JD_CATCHMENT, JD_START, JD_END, JD_MEMBERS)

da_x, da_y = xr.align(da_x, da_y, join="inner")
da_x = da_x.transpose("member", "time")
da_y = da_y.transpose("member", "time")

doy_flat = np.tile(da_x["time"].dt.dayofyear.values, da_x.sizes["member"])
x_flat   = da_x.values.reshape(-1)
y_flat   = da_y.values.reshape(-1)
ok = np.isfinite(x_flat) & np.isfinite(y_flat)
print(f"{int(ok.sum()):,} points "
      f"({da_x.sizes['member']} members × {da_x.sizes['time']} dates)")

_fname = (f"joint_distribution_{cfg.acc_tag(JD_WINDOW_DAYS)}_{x_var}_{y_var}_"
          f"{JD_CATCHMENT}_{JD_START}-{JD_END}.pdf")
make_joint_distribution_figure(
    x_vals=x_flat[ok], y_vals=y_flat[ok], doy_vals=doy_flat[ok],
    x_variable=x_var, y_variable=y_var,
    window_days=JD_WINDOW_DAYS, start_year=JD_START, end_year=JD_END,
    catchment_title=cfg.COMPOUND_CATCHMENTS.get(JD_CATCHMENT, JD_CATCHMENT),
    n_members=da_x.sizes["member"],
    out_paths=figp(_fname),)

985,500 points (90 members × 10950 dates)
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/joint_distribution_2day_precipitation_soil_moisture_regine_drammen_glomma_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/joint_distribution_2day_precipitation_soil_moisture_regine_drammen_glomma_1995-2024.pdf


### Joint-Distribution Analysis for Precip - Soil Moisture and Precip. - Snowmelt, with threshold line


In [6]:
# Joint-distribution scatterplot for CESM2-LE catchments and threshold line
# Identical figure as from the cell above plus the line  x/max(x) + y/max(y) = JD_THRESHOLD.
# The arrays and the whole JD_* selection are reused from the cell above -> only difference is threshold line
# Saved next to the plain PDF with the threshold in the filename: ..._thr0.9.pdf

from catchment_tools import compound_threshold_stats
from plot_style import make_joint_distribution_figure

# ── Selection ─────────────────────────────────────────────────────────────────
JD_CATCHMENT     = "regine_drammen_glomma"        # regine_drammen | regine_glomma | regine_drammen_glomma
JD_COMBO         = ("precipitation", "soil_moisture")  # (x, y); also ("precipitation", "snowmelt") or ("precipitation", "soil_moisture") or ("soil_moisture", "precipitation") or ("soil_moisture", "snowmelt") or ("snowmelt", "soil_moisture")
JD_WINDOW_DAYS   = 2                              # select 2-4
JD_START, JD_END = 1995, 2024                     # inside the stored time-period (1920–2034)
JD_MEMBERS       = "all"                          # "all" | "1-30" | [3, 4, 5, 27, 35] or whatever members
JD_THRESHOLD = 1.25      # x/max(x) + y/max(y) >= JD_THRESHOLD; any value > 0 (max possible = 2.0)

_needed = ("x_flat", "y_flat", "doy_flat", "ok", "x_var", "y_var", "da_x",
           "JD_CATCHMENT", "JD_WINDOW_DAYS", "JD_START", "JD_END")
if any(_n not in globals() for _n in _needed):
    raise NameError(
        "Run the joint-distribution cell directly above first — this cell reuses its "
        "loaded arrays (x_flat / y_flat / doy_flat / ok) and its JD_* selection "
        "(JD_CATCHMENT / JD_COMBO / JD_WINDOW_DAYS / JD_START / JD_END / JD_MEMBERS), "
        "so both figures are guaranteed to show exactly the same points.")

thr = compound_threshold_stats(x_flat[ok], y_flat[ok], JD_THRESHOLD)
print(f"Absolute threshold {JD_THRESHOLD:g} | max {x_var} = {thr['x_max']:.2f}, "
      f"max {y_var} = {thr['y_max']:.2f}")
print(f"  line runs from (0, {thr['y_at_x0']:.2f}) to ({thr['x_at_y0']:.2f}, 0)")
print(f"  {thr['n_exceed']:,} of {thr['n_total']:,} points "
      f"({100.0 * thr['frac_exceed']:.3f} %) satisfy the criterion.")

_fname_thr = (f"joint_distribution_{cfg.acc_tag(JD_WINDOW_DAYS)}_{x_var}_{y_var}_"
              f"{JD_CATCHMENT}_{JD_START}-{JD_END}_thr{JD_THRESHOLD:g}.pdf")
make_joint_distribution_figure(
    x_vals=x_flat[ok], y_vals=y_flat[ok], doy_vals=doy_flat[ok],
    x_variable=x_var, y_variable=y_var,
    window_days=JD_WINDOW_DAYS, start_year=JD_START, end_year=JD_END,
    catchment_title=cfg.COMPOUND_CATCHMENTS.get(JD_CATCHMENT, JD_CATCHMENT),
    n_members=da_x.sizes["member"],
    threshold=JD_THRESHOLD,
    x_norm_max=thr["x_max"], y_norm_max=thr["y_max"],
    out_paths=figp(_fname_thr),
)

Absolute threshold 1.25 | max precipitation = 83.89, max soil_moisture = 693.38
  line runs from (0, 866.73) to (104.86, 0)
  1,359 of 985,500 points (0.138 %) satisfy the criterion.
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/joint_distribution_2day_precipitation_soil_moisture_regine_drammen_glomma_1995-2024_thr1.25.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/joint_distribution_2day_precipitation_soil_moisture_regine_drammen_glomma_1995-2024_thr1.25.pdf


### Compound Extreme Frequency Evolution - Selection, Validation and Computation


In [22]:
# Rolling-window time evolution of the compound threshold  s = x/max(x) + y/max(y) >= FE_THRESHOLD
# Reuses the caches, N-day operators and severity definition of the joint-distribution cells above.
# The only difference: max(x)/max(y) are FROZEN on FE_NORM_REF *and* on FE_SEASON, so the criterion
# is ONE fixed line in the (x, y) plane for every rolling window — the reference maxima come from
# exactly the months the exceedances are later counted in, and a drifting denominator cannot fake
# a change. Accepted limitations: no declustering (one storm can exceed on up to FE_WINDOW_DAYS
# consecutive days, so an "event" = an exceedance day) and overlapping rolling windows.

from catchment_tools import (run_compound_frequency_evolution, print_frequency_evolution_summary,
                             write_frequency_evolution_outputs, season_label)

# ── Selection ─────────────────────────────────────────────────────────────────
FE_CATCHMENT      = "regine_drammen_glomma"        # regine_drammen | regine_glomma | regine_drammen_glomma
FE_COMBO          = ("precipitation", "snowmelt")  # ordered pair of: precipitation | snowmelt | soil_moisture
FE_WINDOW_DAYS    = 2                              # N-day event definition (1-4; >= 2 with snowmelt)
FE_START, FE_END  = 1920, 2034                     # analysis period, inside the stored record (1920–2034)
FE_MEMBERS        = "all"                          # "all" | "20-60" | [2, 4, 6, 8]  (90 members exist)
FE_THRESHOLD      = 0.9                           # s* ; any value > 0 (max possible = 2.0)
FE_ROLL_YEARS     = 10                             # length L of the centred rolling window
FE_ROLL_STEP      = 1                              # years between windows
FE_SEASON         = "MAMJ"                         # all | DJF | MAM | JJA | SON | MAMJ | (11, 4)

FE_NORM_REF       = (1995, 2024)                   # reference period for threshold
FE_NORM_REF_MEM   = "all"                          # reference members

# ── Figure options ────────────────────────────────────────────────────────────
# Figure spread: "std" = blue ±1 standard-deviation band
# Figure spread: "p025p975" = 2.5-97.5%-tile
# Figure spread: "minmax" = min and max. members
FE_SPREAD_SHOW    = ("std", "minmax")              # change to: "std", "minmax", "p025p975"
FE_SAVE_CSV       = False                          # ensemble table + metadata JSON next to the PDFs

FE = run_compound_frequency_evolution(dict(
    catchment=FE_CATCHMENT, combo=FE_COMBO, window_days=FE_WINDOW_DAYS,
    start_year=FE_START, end_year=FE_END, members=FE_MEMBERS, threshold=FE_THRESHOLD,
    roll_years=FE_ROLL_YEARS, roll_step=FE_ROLL_STEP, season=FE_SEASON,
    norm_ref=FE_NORM_REF, norm_ref_members=FE_NORM_REF_MEM,
    spread_show=FE_SPREAD_SHOW))
print_frequency_evolution_summary(FE)

_fe_c, _fe_e, _fe_d = FE["config"], FE["ensemble"], FE["diagnostics"]
fe_x_var, fe_y_var = _fe_c["combo"]
_fe_args = dict(window_days=_fe_c["window_days"], x_variable=fe_x_var, y_variable=fe_y_var,
                catchment_slug=_fe_c["catchment"], start_year=_fe_c["start_year"],
                end_year=_fe_c["end_year"], roll_years=_fe_c["roll_years"],
                threshold=_fe_c["threshold"], norm_ref=_fe_c["norm_ref"],
                season=_fe_c["season_tag"], members=_fe_c["members"])
_fe_stem1 = cfg.compound_freq_stem("internal_variability_trend", **_fe_args)
_fe_stem2 = cfg.compound_freq_stem("signal_to_noise", **_fe_args)

if FE_SAVE_CSV:
    write_frequency_evolution_outputs(FE, _fe_stem1, cfg.compound_freq_figure_paths)

[ok] 2-day caches present: post_processed_cesm2_le_2day_regine_drammen_glomma_precipitation_1920-2034.nc, post_processed_cesm2_le_2day_regine_drammen_glomma_snowmelt_1920-2034.nc
COMPOUND FREQUENCY EVOLUTION — precipitation + snowmelt | regine_drammen_glomma | 2-day | 1920–2034 | season MAMJ
Frozen maxima (ref 1995–2024, season MAMJ, 90 members): max precipitation = 83.885, max snowmelt = 26.229
Physical threshold: precipitation/83.89 + snowmelt/26.23 >= 0.9  → intercepts (75.50, 0) and (0, 23.61)
Candidate days: 1,262,700 | exceedance days (= events): 306
Windows: 106 complete 10-year windows, centres 1924.5–2029.5 (step 1 yr)
First window 1924.5: 0.04556 events/year   |   last window 2029.5: 0.02889 events/year
Mean σ across members: 0.06074 events/year
S/N (ensemble mean / σ): range 0.37 … 0.61   |   first window 0.53 → last window 0.49
[warning] only 0.3 events per member per 10-year window — σ across members is dominated by counting noise. Lengthen FE_ROLL_YEARS before quoting σ q

### Compound Hazard Frequency and Internal Variability over Time


In [23]:
# Compound-extreme hazard frequency (events per year) per centred rolling window:
# ensemble mean, the blue ±1 standard-deviation band and the dashed ensemble spread
# (min/max member OR 2.5/97.5 %, chosen with FE_SPREAD_SHOW in the cell above).
# Reuses the arrays computed in that cell, so both figures describe exactly the same events.
from plot_style import plot_internal_variability_trend, frequency_selection_lines


FE_SELECTION = frequency_selection_lines(
    x_variable=fe_x_var, y_variable=fe_y_var, threshold=_fe_c["threshold"],
    window_days=_fe_c["window_days"], roll_years=_fe_c["roll_years"],
    n_members=_fe_d["n_members"], norm_ref=_fe_c["norm_ref"],
    season_label=None if _fe_c["season_months"] is None else season_label(_fe_c["season"]))

plot_internal_variability_trend(
    _fe_e["window_centre"].to_numpy(), _fe_e["f_mean"].to_numpy(),
    _fe_e["sigma"].to_numpy(), _fe_e["min"].to_numpy(), _fe_e["max"].to_numpy(),
    x_variable=fe_x_var, y_variable=fe_y_var,
    catchment_title=cfg.COMPOUND_CATCHMENTS.get(_fe_c["catchment"], _fe_c["catchment"]),
    start_year=_fe_c["start_year"], end_year=_fe_c["end_year"],
    roll_years=_fe_c["roll_years"], selection_lines=FE_SELECTION,
    spread_show=_fe_c["spread_show"],
    p025=_fe_e["p025"].to_numpy(), p975=_fe_e["p975"].to_numpy(),
    out_paths=cfg.compound_freq_figure_paths(f"{_fe_stem1}.pdf"),)

Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/frequency_evolution/internal_variability_trend_2day_precipitation_snowmelt_regine_drammen_glomma_1920-2034_10year_thr0.9_ref1995-2024_MAMJ.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/frequency_evolution/internal_variability_trend_2day_precipitation_snowmelt_regine_drammen_glomma_1920-2034_10year_thr0.9_ref1995-2024_MAMJ.pdf


### Signal-to-Noise Ratio of Compound Hazard Frequency


In [24]:
# Signal-to-noise ratio (ensemble mean / standard deviation across members) of exactly
# the same rolling-window rates as Figure 1: same threshold, N-day window, member pool, frozen
# reference and season. Reuses FE_SELECTION from the cell above, so both legend blocks are identical
from plot_style import plot_signal_to_noise_ratio

plot_signal_to_noise_ratio(
    _fe_e["window_centre"].to_numpy(), _fe_e["signal_to_noise"].to_numpy(),
    x_variable=fe_x_var, y_variable=fe_y_var,
    catchment_title=cfg.COMPOUND_CATCHMENTS.get(_fe_c["catchment"], _fe_c["catchment"]),
    start_year=_fe_c["start_year"], end_year=_fe_c["end_year"],
    selection_lines=FE_SELECTION,
    out_paths=cfg.compound_freq_figure_paths(f"{_fe_stem2}.pdf"),)

Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/frequency_evolution/signal_to_noise_2day_precipitation_snowmelt_regine_drammen_glomma_1920-2034_10year_thr0.9_ref1995-2024_MAMJ.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/frequency_evolution/signal_to_noise_2day_precipitation_snowmelt_regine_drammen_glomma_1920-2034_10year_thr0.9_ref1995-2024_MAMJ.pdf


### Threshold Exceedance Count

In [12]:
# How many (member, date) points lie above the frozen threshold line inside FE_NORM_REF itself.
# Same frozen x_max/y_max, same season and same reference member pool as the cell above.
from catchment_tools import (load_compound_pair, parse_member_selection,
                             subset_time_series_by_year, compound_threshold_stats)

_rx, _ry = load_compound_pair(_fe_c["catchment"], _fe_c["combo"], _fe_c["window_days"])
_mem     = parse_member_selection(_fe_c["norm_ref_members"], list(_rx["member"].values))
_rx = subset_time_series_by_year(_rx.sel(member=_mem), *_fe_c["norm_ref"])
_ry = subset_time_series_by_year(_ry.sel(member=_mem), *_fe_c["norm_ref"])
if _fe_c["season_months"] is not None:
    _keep   = np.isin(_rx["time"].dt.month.values, _fe_c["season_months"])
    _rx, _ry = _rx.isel(time=_keep), _ry.isel(time=_keep)

_ref = compound_threshold_stats(_rx.values.ravel(), _ry.values.ravel(),
                                _fe_c["threshold"], x_max=_fe_d["x_max"], y_max=_fe_d["y_max"])

print(f"Reference window {_fe_c['norm_ref'][0]}–{_fe_c['norm_ref'][1]} | season "
      f"{_fe_c['season_tag']} | {len(_mem)} members | threshold {_fe_c['threshold']:g}")
print(f"  {_ref['n_exceed']:,} of {_ref['n_total']:,} points above the line "
      f"({100.0 * _ref['frac_exceed']:.3f} %)  →  "
      f"{_ref['n_exceed'] / len(_mem) / (_fe_c['norm_ref'][1] - _fe_c['norm_ref'][0] + 1):.3f} "
      "events per member per year")
print(f"  for comparison, analysis period {_fe_c['start_year']}–{_fe_c['end_year']}: "
      f"{_fe_d['n_exceedance_days']:,} of {_fe_d['n_candidate_days']:,} points "
      f"({100.0 * _fe_d['n_exceedance_days'] / _fe_d['n_candidate_days']:.3f} %)")


Reference window 1995–2024 | season MAMJ | 90 members | threshold 0.75
  594 of 329,400 points above the line (0.180 %)  →  0.220 events per member per year
  for comparison, analysis period 1920–2034: 2,726 of 1,262,700 points (0.216 %)
